# Milestone 2 - modelling a problem, and looking one up

**Why this matters.** Interviewers watch how you model data before you write
logic. A good dataclass makes the rest of the code obvious; a dict-of-dicts makes
every later function defensive. This milestone is small on purpose - the lesson is
in the details, especially the error message.

**How to work through this notebook.** Run the cells top to bottom. Each
exercise has a cell for you to fill in, then a cell of tests. Run the tests
first and watch them fail - read *what* they check, because the tests are the
specification. Then write code until they go green.

After you edit an exercise cell, **re-run it** before re-running its tests - the
tests use whatever is currently defined in the notebook.

In [ ]:
import ipytest
import pytest

# clean="test_*": before each test cell, forget test functions from earlier cells
# - but not other names starting with "Test", like milestone 2's TestCase class.
ipytest.autoconfig(clean="test_*")

import dataclasses
from dataclasses import dataclass
from typing import Any

## 1. The data model

`TestCase` is one input/output pair. `frozen=True` makes instances immutable and
hashable. Keep it: a test case that can be edited after construction is a bug
waiting to happen.

`Problem` is one interview problem. It needs two properties on top of its fields:
`samples` (only the cases with `is_sample` true) and `hidden` (the rest). Why
both: samples teach the format, hidden cases stop you passing by special-casing
the examples. An interviewer holds some tests back too.

In [ ]:
@dataclass(frozen=True)
class TestCase:
    """One input/output pair.

    TODO: give this three fields:
        args      tuple[Any, ...]  - positional arguments to call the function with
        expected  Any              - what it must return
        is_sample bool             - shown to the learner up front; default False
    """

    # Tells pytest this is a data class, not a test class to collect.
    # Without it, the name "TestCase" trips pytest's test-discovery heuristic.
    __test__ = False


@dataclass(frozen=True)
class Problem:
    """One interview problem.

    TODO: give this at least these fields:
        id             str
        title          str
        topic          str
        difficulty     str
        function_name  str
        test_cases     tuple[TestCase, ...]

    Then add two properties:
        samples -> tuple[TestCase, ...]   only the cases with is_sample True
        hidden  -> tuple[TestCase, ...]   only the cases with is_sample False
    """

In [ ]:
# A helper the tests below use to build problems quickly.
def make(pid: str, topic: str = "hash-map", difficulty: str = "easy") -> Problem:
    return Problem(
        id=pid, title=pid.title(), topic=topic, difficulty=difficulty,
        function_name=pid,
        test_cases=(
            TestCase(args=(1,), expected=1, is_sample=True),
            TestCase(args=(2,), expected=2),
        ),
    )

In [ ]:
%%ipytest -qq

def test_testcase_defaults_to_hidden():
    assert TestCase(args=(1,), expected=1).is_sample is False


def test_dataclasses_are_frozen():
    case = TestCase(args=(1,), expected=1)
    with pytest.raises(dataclasses.FrozenInstanceError):
        case.expected = 2


def test_samples_and_hidden_partition_the_cases():
    problem = make("two_sum")
    assert len(problem.samples) == 1 and problem.samples[0].is_sample
    assert len(problem.hidden) == 1 and not problem.hidden[0].is_sample
    assert len(problem.samples) + len(problem.hidden) == len(problem.test_cases)

## 2. `build_index`

Build an id -> `Problem` lookup table. Raise `ValueError` if two problems share an
id - a duplicate would silently shadow one of them, and you would lose an
afternoon to it.

In [ ]:
def build_index(problems: tuple[Problem, ...]) -> dict[str, Problem]:
    """Build an id -> Problem lookup table."""
    # TODO: implement.
    raise NotImplementedError("Milestone 2: implement build_index")

In [ ]:
%%ipytest -qq

def test_index_maps_id_to_problem():
    problems = (make("a"), make("b"))
    index = build_index(problems)
    assert index["a"] is problems[0] and index["b"] is problems[1]


def test_duplicate_ids_are_rejected():
    with pytest.raises(ValueError):
        build_index((make("a"), make("a")))

## 3. `get_problem`

Look up one problem by id. On a miss, raise `KeyError` whose message lists the
available ids. Compare:

```
KeyError: 'two-sum'
KeyError: "No problem with id 'two-sum'. Available: two_sum, binary_search, ..."
```

The second one costs you two extra lines and saves every future user - you
included - a trip into the source. Error messages are a design surface.

In [ ]:
def get_problem(index: dict[str, Problem], problem_id: str) -> Problem:
    """Look up one problem by id."""
    # TODO: implement.
    raise NotImplementedError("Milestone 2: implement get_problem")

In [ ]:
%%ipytest -qq

def test_lookup_finds_a_problem():
    index = build_index((make("two_sum"),))
    assert get_problem(index, "two_sum").id == "two_sum"


def test_lookup_miss_raises_keyerror():
    with pytest.raises(KeyError):
        get_problem(build_index((make("two_sum"),)), "nope")


def test_lookup_miss_lists_the_valid_ids():
    """The whole point of milestone 2: a message that saves the next person a trip."""
    index = build_index((make("two_sum"), make("binary_search")))
    with pytest.raises(KeyError) as exc:
        get_problem(index, "two-sum")
    message = str(exc.value)
    assert "two_sum" in message and "binary_search" in message

## 4. `filter_problems`

Filter by topic and/or difficulty. Both arguments are optional and combine with
AND. Passing neither returns everything. Return a tuple, not a list.

In [ ]:
def filter_problems(
    problems: tuple[Problem, ...],
    topic: str | None = None,
    difficulty: str | None = None,
) -> tuple[Problem, ...]:
    """Filter by topic and/or difficulty."""
    # TODO: implement.
    raise NotImplementedError("Milestone 2: implement filter_problems")

In [ ]:
%%ipytest -qq

def test_filter_by_topic():
    problems = (make("a", topic="stack"), make("b", topic="graph"))
    assert filter_problems(problems, topic="stack") == (problems[0],)


def test_filter_by_difficulty():
    problems = (make("a", difficulty="easy"), make("b", difficulty="medium"))
    assert filter_problems(problems, difficulty="medium") == (problems[1],)


def test_filters_combine_with_and():
    problems = (make("a", topic="stack", difficulty="easy"),
                make("b", topic="stack", difficulty="medium"))
    assert filter_problems(problems, topic="stack", difficulty="medium") == (problems[1],)


def test_no_filter_returns_everything_as_a_tuple():
    problems = (make("a"), make("b"))
    assert filter_problems(problems) == problems


def test_no_match_returns_an_empty_tuple():
    assert filter_problems((make("a"),), topic="nope") == ()

## Try it

Feel the difference a good error message makes.

In [ ]:
index = build_index((make("two_sum"), make("binary_search"), make("num_islands")))
get_problem(index, "two-sum")